# Пример - Проверка тождеств симметрических полиформ

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **73ac82d**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/73ac82d5c16afbfd47285436e1b6f37d5a68f8af.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Начнём с тождества на пяти вершинах. Затем проверим нормировку типов, произведения форм, экспоненты графов и переход к многочленам Абеля. Расчёты относятся к указанным размерам и грейдам. Общие доказательства даны в заметках серии.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra 73ac82d](https://github.com/magin67/polyform_algebra/commit/73ac82d5c16afbfd47285436e1b6f37d5a68f8af).


## Тождество на пяти вершинах

На вершинах $a,b,c,d,e$ построим левую часть из десяти разбиений на тройку и дополняющую пару. Справа сложим формы пяти четвёрок и умножим сумму на $3$. Оставшаяся вершина в каждом разбиении справа образует одноэлементный блок.

$$
E_5[2,1]=3E_5[3,0]
$$

Функция `q` строит форму разбиения из его блоков, а вызов без блоков возвращает единицу $e$. Функция `check` сравнивает коэффициенты обеих форм в одном контексте. При расхождении она показывает пару базисных внешних объектов и коэффициент разности. Для контроля сравним левую часть ещё и с $E_5[3,0]$ без множителя $3$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B/%D1%84%D1%83%D0%BD%D0%B4%D0%B0%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F-%D1%82%D0%B5%D0%BE%D1%80%D0%B5%D0%BC%D0%B0-%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D1%85-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BF%D0%B5%D1%80%D0%B5%D0%B4-%D0%BE%D0%B1%D1%89%D0%B5%D0%B9-%D1%84%D0%BE%D1%80%D0%BC%D1%83%D0%BB%D0%BE%D0%B9).

In [3]:
from itertools import combinations

from polyform_algebra.core import (
    Context, equivalent, form_coefficients, partition_form,
)


def q(context, *blocks):
    return partition_form(context, blocks).value


def principal(context, grade):
    return sum((q(context, block)
                for block in combinations(context.labels, grade + 1)),
               start=0 * q(context))


def check(identity, left, right):
    matches = equivalent(left, right).value
    if not matches:
        pair, difference = form_coefficients(left - right).value[0]
        blades = tuple(" ∧ ".join(left.context.labels[i] for i in blade)
                       for blade in pair)
        raise AssertionError((identity, blades, difference))
    print(identity, "→", matches)


five = Context(tuple("abcde"))
triple_pair_terms = [
    q(five, triple, tuple(vertex for vertex in five.labels if vertex not in triple))
    for triple in combinations(five.labels, 3)
]
assert len(triple_pair_terms) == 10
assert len(list(combinations(five.labels, 4))) == 5
E5_21 = sum(triple_pair_terms, start=0 * q(five))
E5_30 = principal(five, 3)
check("E₅[2,1] = 3 E₅[3,0]", E5_21, 3 * E5_30)
assert not equivalent(E5_21, E5_30).value

E₅[2,1] = 3 E₅[3,0] → True


## Типы и нормировки

Перечислим разбиения, добавляя первую вершину либо отдельным блоком, либо к одному из блоков разбиения оставшихся вершин. Каждое разбиение получается один раз. Размер блока минус единица даёт его грейд. Суммируем формы по типу и сверяем число слагаемых с формулой размера представления.

На четырёх вершинах три разбиения на пары дают ту же форму, что и четыре разбиения с тройкой. Для каждого из $23$ типов положительного грейда при $2\leqslant n\leqslant6$ сравним сумму форм разбиений с кратной ей главной полиформой. Множитель вычисляется по фундаментальной теореме через факториалы и кратности равных положительных грейдов. Отдельно выведем три тождества на шести вершинах, включая тип с одноэлементным блоком.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B---%D1%82%D0%B8%D0%BF%D1%8B-%D0%B8-%D0%BD%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%BA%D0%B8#%D1%82%D0%B8%D0%BF-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B-%D0%B8-%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F-%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B0).

In [4]:
from collections import Counter
from fractions import Fraction
from math import factorial, prod


def set_partitions(labels):
    if not labels:
        yield ()
        return
    first, *rest = labels
    for blocks in set_partitions(rest):
        yield ((first,),) + blocks
        for index, block in enumerate(blocks):
            yield blocks[:index] + ((first,) + block,) + blocks[index + 1:]


def block_type(blocks):
    return tuple(sorted((len(block) - 1 for block in blocks), reverse=True))


def group_forms(context):
    sums = {}
    counts = Counter()
    zero = 0 * q(context)
    for blocks in set_partitions(context.labels):
        grades = block_type(blocks)
        sums[grades] = sums.get(grades, zero) + q(context, *blocks)
        counts[grades] += 1
    return sums, counts


def type_count(n, grades):
    divisor = prod(factorial(grade + 1) for grade in grades)
    repeats = prod(factorial(count) for count in Counter(grades).values())
    return Fraction(factorial(n), divisor * repeats)


def principal_factor(grades):
    positive = tuple(grade for grade in grades if grade)
    repeats = prod(factorial(count) for count in Counter(positive).values())
    divisor = factorial(grades.count(0)) * prod(map(factorial, positive)) * repeats
    return Fraction(factorial(len(grades) - 1) * factorial(sum(grades)), divisor)


four = Context(tuple("abcd"))
six = Context(tuple("abcdef"))
four_types, four_counts = group_forms(four)
E4_2 = principal(four, 2)
E4_11 = four_types[(1, 1)]
assert four_counts[(1, 1)] == 3 and four_counts[(2, 0)] == 4
check("E₄[1,1] = E₄[2,0]", E4_11, E4_2)

checked = 0
for n in range(2, 7):
    context = Context(tuple("abcdef"[:n]))
    type_sums, type_counts = group_forms(context)
    for grades, form in type_sums.items():
        grade = sum(grades)
        assert type_counts[grades] == type_count(n, grades), grades
        if grade:
            right = principal_factor(grades) * principal(context, grade)
            assert equivalent(form, right).value, (n, grades)
            checked += 1

six_types, six_counts = group_forms(six)
for grades, coefficient in (((1, 1, 1), 2), ((2, 1, 0), 6), ((2, 2), 3)):
    assert principal_factor(grades) == coefficient
    check(f"E₆{list(grades)} = {coefficient} E₆[{sum(grades)}]",
          six_types[grades], coefficient * principal(six, sum(grades)))
assert checked == 23
print("Positive-grade types (2 ≤ n ≤ 6):", checked)

E₄[1,1] = E₄[2,0] → True


E₆[1, 1, 1] = 2 E₆[3] → True
E₆[2, 1, 0] = 6 E₆[3] → True
E₆[2, 2] = 3 E₆[4] → True
Positive-grade types (2 ≤ n ≤ 6): 23


## Произведения и степени

Левые части строим умножением форм в пакете. Для квадрата $E_4[1]$ и произведений на пяти и шести вершинах справа сначала используем разложение по типам блоков, затем выражаем сумму через главную полиформу. Проверим также куб $E_4[1]$ и исчезновение степеней, грейд которых превышает $n-1$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B/%D0%BF%D1%80%D0%BE%D0%B8%D0%B7%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D1%82%D0%B5%D0%BF%D0%B5%D0%BD%D0%B8-%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D1%85-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%BD%D0%B0-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85-%D0%BF%D1%8F%D1%82%D0%B8-%D0%B8-%D1%88%D0%B5%D1%81%D1%82%D0%B8-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [5]:
from polyform_algebra.core import form_product


def multiply(left, right):
    return form_product(left, right).value


E4_1 = principal(four, 1)
E4_3 = principal(four, 3)
square4 = multiply(E4_1, E4_1)
cube4 = multiply(square4, E4_1)
check("E₄[1]² = 6 E₄[2] + 2 E₄[1,1]", square4, 6 * E4_2 + 2 * E4_11)
check("E₄[1]² = 8 E₄[2]", square4, 8 * E4_2)
check("E₄[1]³ = 96 E₄[3]", cube4, 96 * E4_3)
check("E₄[1]⁴ = 0", multiply(cube4, E4_1), 0 * E4_1)

product5 = multiply(principal(five, 1), principal(five, 2))
check("E₅[1] E₅[2] = 12 E₅[3] + E₅[2,1]", product5, 12 * E5_30 + E5_21)
check("E₅[1] E₅[2] = 15 E₅[3]", product5, 15 * E5_30)

E6_2 = principal(six, 2)
E6_4 = principal(six, 4)
square6 = multiply(E6_2, E6_2)
check("E₆[2]² = 30 E₆[4] + 2 E₆[2,2]", square6, 30 * E6_4 + 2 * six_types[(2, 2)])
check("E₆[2]² = 36 E₆[4]", square6, 36 * E6_4)
check("E₆[2]³ = 0", multiply(square6, E6_2), 0 * E6_2)

E₄[1]² = 6 E₄[2] + 2 E₄[1,1] → True
E₄[1]² = 8 E₄[2] → True
E₄[1]³ = 96 E₄[3] → True
E₄[1]⁴ = 0 → True
E₅[1] E₅[2] = 12 E₅[3] + E₅[2,1] → True
E₅[1] E₅[2] = 15 E₅[3] → True


E₆[2]² = 30 E₆[4] + 2 E₆[2,2] → True
E₆[2]² = 36 E₆[4] → True


E₆[2]³ = 0 → True


## Два разложения экспоненты полного графа

Возьмём полный граф $K_5$ с единичными проводимостями. Слева вычислим полиформенную экспоненту его лапласиана. Справа построим две суммы. В первой сгруппируем формы по типу леса и взвесим их числом выборов деревьев на заданных блоках. Во второй возьмём главную форму грейда $k$ с коэффициентом $5^{k-1}$ и отдельно добавим $e$.

Сравним обе суммы с экспонентой целиком, затем проверим формулы для второго и третьего грейдов по отдельности. Старший грейд даёт остовной коэффициент $\tau(K_5)=125$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BF%D0%BE%D0%BB%D0%BD%D0%BE%D0%B3%D0%BE-%D0%B3%D1%80%D0%B0%D1%84%D0%B0#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%B3%D1%80%D0%B0%D1%84-%D0%BD%D0%B0-%D0%BF%D1%8F%D1%82%D0%B8-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B0%D1%85).

In [6]:
from polyform_algebra.core import (
    Network, form_exp, grade_part, laplacian_form, spanning_coefficient,
)


def complete_graph(context):
    return Network(context, tuple((a, b, 1) for a, b in combinations(context.labels, 2)))


def tree_factor(grades):
    return prod((grade + 1) ** (grade - 1) for grade in grades if grade)


def principal_exp(context):
    n = len(context.labels)
    return sum((n ** (k - 1) * principal(context, k) for k in range(1, n)),
               start=q(context))


K5 = complete_graph(five)
M_K5 = form_exp(laplacian_form(K5).value).value
five_types, _ = group_forms(five)
assert (tree_factor((2, 0, 0)), tree_factor((2, 1))) == (3, 3)
forest_exp = sum((tree_factor(grades) * form for grades, form in five_types.items()),
                 start=0 * q(five))
check("M(K₅) = forest expansion", M_K5, forest_exp)
check("M(K₅) = principal expansion", M_K5, principal_exp(five))
check("M₂(K₅) = 3 E₅[2] + E₅[1,1]", grade_part(M_K5, 2).value,
      3 * principal(five, 2) + five_types[(1, 1, 0)])
check("M₃(K₅) = 16 E₅[3] + 3 E₅[2,1]", grade_part(M_K5, 3).value,
      16 * E5_30 + 3 * E5_21)
tau_K5 = spanning_coefficient(M_K5).value
assert tau_K5 == 125
print("τ(K₅) =", tau_K5)

M(K₅) = forest expansion → True
M(K₅) = principal expansion → True
M₂(K₅) = 3 E₅[2] + E₅[1,1] → True
M₃(K₅) = 16 E₅[3] + 3 E₅[2,1] → True


τ(K₅) = 125


## Удаление диагоналей

Построим $M_{C_4}$ произведением четырёх рёберных множителей $e+[(uv)]^2$ для цикла $a-b-c-d-a$ с единичными проводимостями. Обозначим формы диагоналей $ac$ и $bd$ через $p$ и $q$. Справа начнём с главного разложения $M_{K_4}$ и удалим диагонали умножением на $(e-p)(e-q)$. Проверим и полученную полиформу, и её сокращённую запись из заметки. Из старшего грейда извлечём $\tau(C_4)=4$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BF%D0%BE%D0%BB%D0%BD%D0%BE%D0%B3%D0%BE-%D0%B3%D1%80%D0%B0%D1%84%D0%B0#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D1%83%D0%B4%D0%B0%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B4%D0%B8%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D0%B5%D0%B9-%D1%87%D0%B5%D1%82%D1%8B%D1%80%D1%91%D1%85%D1%86%D0%B8%D0%BA%D0%BB%D0%B0).

In [7]:
cycle_edges = (("a", "b"), ("b", "c"), ("c", "d"), ("d", "a"))
cycle_product = q(four)
for edge in cycle_edges:
    cycle_product = multiply(cycle_product, q(four) + q(four, edge))

p = q(four, ("a", "c"))
diagonal_q = q(four, ("b", "d"))
diagonals = multiply(p, diagonal_q)
deleted = multiply(principal_exp(four), q(four) - p - diagonal_q + diagonals)
cycle_formula = q(four) + (E4_1 - p - diagonal_q) + 2 * E4_2 - diagonals + 4 * E4_3
check("M(C₄) = M(K₄) (e − p) (e − q)", cycle_product, deleted)
check("M(C₄) = simplified expansion", cycle_product, cycle_formula)
tau_C4 = spanning_coefficient(cycle_product).value
assert tau_C4 == 4
print("τ(C₄) =", tau_C4)

M(C₄) = M(K₄) (e − p) (e − q) → True
M(C₄) = simplified expansion → True
τ(C₄) = 4


## Лесные коэффициенты и многочлены Абеля

На вспомогательном множестве из $k$ буквенных вершин перечислим разбиения на $m$ блоков. К каждому блоку мысленно добавим один корень. Число деревьев на расширенном блоке учитывается множителем Кэли. Сумма по разбиениям даёт $\operatorname{St}(k,m)$.

Слева построим $A_k(x)=x(x+k)^{k-1}$, справа — сумму $\sum_m\operatorname{St}(k,m)x_{\underline m}$. Сравним целочисленные коэффициенты при всех степенях $x$ для $1\leqslant k\leqslant6$. Для $A_3(2)$ отдельно вычислим определитель минора лапласиана $K_5$, удалив строки и столбцы двух заданных корней $d,e$. Обе записи дают $50$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B0%D0%BB%D0%B3%D0%B5%D0%B1%D1%80%D0%B0-%D0%B8-%D0%BA%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B0%D1%82%D0%BE%D1%80%D0%B8%D0%BA%D0%B0-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC/%D1%81%D0%B8%D0%BC%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D1%84%D0%BE%D1%80%D0%BC%D1%8B/%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B-%D0%B8-%D0%BC%D0%BD%D0%BE%D0%B3%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D1%8B-%D0%B0%D0%B1%D0%B5%D0%BB%D1%8F#%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BC%D0%BD%D0%BE%D0%B3%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%BE%D0%B2-%D0%B0%D0%B1%D0%B5%D0%BB%D1%8F).

In [8]:
from sympy import Matrix, Poly, Symbol
from polyform_algebra.core import laplacian

x = Symbol("x")
for k in range(1, 7):
    St = Counter()
    for blocks in set_partitions(tuple("abcdef"[:k])):
        # Adding a root to each block makes its size a boundary grade.
        grades = tuple(len(block) for block in blocks)
        St[len(blocks)] += tree_factor(grades)
    abel = Poly(x * (x + k) ** (k - 1), x)
    falling = Poly(sum(St[m] * prod(x - j for j in range(m))
                       for m in range(1, k + 1)), x)
    assert abel == falling, (k, abel - falling)
    print(f"A{k}(x) =", falling.as_expr())
    if k == 3:
        assert tuple(St[m] for m in range(1, 4)) == (16, 9, 1)
        assert abel.eval(2) == 50
    if k == 4:
        assert tuple(St[m] for m in range(1, 5)) == (125, 91, 18, 1)

# The principal minor keeps ordinary vertices a, b, c and deletes roots d, e.
ordinary = [K5.context.labels.index(vertex) for vertex in "abc"]
minor = Matrix(laplacian(K5).value.values).extract(ordinary, ordinary)
assert minor.det() == 2 * (3 + 2) ** (3 - 1) == 50
print("A3(2) = det(L[abc, abc]) =", minor.det())

A1(x) = x
A2(x) = x**2 + 2*x
A3(x) = x**3 + 6*x**2 + 9*x
A4(x) = x**4 + 12*x**3 + 48*x**2 + 64*x
A5(x) = x**5 + 20*x**4 + 150*x**3 + 500*x**2 + 625*x
A6(x) = x**6 + 30*x**5 + 360*x**4 + 2160*x**3 + 6480*x**2 + 7776*x
A3(2) = det(L[abc, abc]) = 50
